# Cap10 Three-Fold Checkpoint Metric Replay V2

This notebook continues the SOXLBacktest/qLib target line after the three-fold BC slice gate passed. It trains fold-local BC warm starts with explicit release-gate weighting, runs the short checkpoint/PPO selector profile, evaluates official walk-forward replay metrics, and blocks promotion unless every required fold clears the user-facing target: relative return >= +30pp and MaxDD < 20%.

This is still diagnostic: no actor is retained here, and a later full retained candidate must still run fixed-split, YTD return alignment, and dashboard replay evidence.

In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.checkpoint_daily_export import (
    build_checkpoint_candidate_daily_export,
    metrics_from_official_backtest,
)
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'wf2020_return_preserving_bull_participation_target_v1'
TARGETS_PATH = SOURCE_CARD / 'return_preserving_bull_participation_three_fold_target_contract.csv'
SOURCE_SELECTION_PATH = SOURCE_CARD / 'summary.json'
REFERENCE_SELECTION_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_reference_model_score_v1' / 'selection.json'
FRAME_SOURCE = 'build_dataset_rebuilt_full_2010_2024'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260613
TARGET_RELATIVE_RETURN = 30.0
TARGET_RELATIVE_RETURN_BY_FOLD = {
    'wf_2020': TARGET_RELATIVE_RETURN,
    'wf_2022': TARGET_RELATIVE_RETURN,
    'wf_2023_2024': TARGET_RELATIVE_RETURN,
}
TARGET_MAX_DD = 20.0
BC_SLICE_ACCURACY_MIN = 0.60
REQUIRED_SLICES = {'release_gate_state': 20, 'is_downtrend_state': 100, 'drawdown_control_state': 20}
DRIFT_RELATIVE_FLOOR_PP = -5.0
DRIFT_MAX_DD_CEILING_PP = 2.0
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2018-12-31'),
        'valid': ('2019-01-01', '2019-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_replay_feature_frame(profile):
    dataset = build_dataset(
        train_seg=('2010-03-11', '2022-12-31'),
        valid_seg=('2023-01-01', '2023-12-31'),
        test_seg=('2024-01-01', '2024-12-31'),
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    parts = [segment_frame(dataset, name) for name in ('train', 'valid', 'test')]
    frame = pd.concat(parts).sort_index()
    return frame.loc[~frame.index.duplicated(keep='last')].copy()

print('root', ROOT)
print('card', CARD)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1


In [2]:
source_selection = json.loads(SOURCE_SELECTION_PATH.read_text())
acceptance_criteria = source_selection.get('acceptance_criteria', {}) if isinstance(source_selection, dict) else {}
TARGET_RELATIVE_RETURN_BY_FOLD.update({
    'wf_2020': float(acceptance_criteria.get('wf2020_relative_return_pp_gte', TARGET_RELATIVE_RETURN)),
    'wf_2022': float(acceptance_criteria.get('wf2022_relative_return_pp_gte', TARGET_RELATIVE_RETURN)),
    'wf_2023_2024': float(acceptance_criteria.get('wf2023_2024_relative_return_pp_gte', TARGET_RELATIVE_RETURN)),
})
print('target_relative_return_by_fold', TARGET_RELATIVE_RETURN_BY_FOLD)
reference_selection = json.loads(REFERENCE_SELECTION_PATH.read_text()) if REFERENCE_SELECTION_PATH.exists() else {}
profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
frame = build_replay_feature_frame(profile)
print('feature_frame_source', FRAME_SOURCE, 'rows', len(frame), 'cols', len(frame.columns))
targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)
    else:
        targets[col] = False

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    valid_start = pd.Timestamp(spec['valid'][0])
    valid_end = pd.Timestamp(spec['valid'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_all = targets[targets['fold'].astype(str).eq(spec['target_fold'])].copy()
    fold_targets = fold_all[fold_all['datetime'].between(train_start, train_end)].copy()
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'all_target_rows': int(len(fold_all)),
        'train_target_rows': int(len(fold_targets)),
        'valid_overlap_rows': int(fold_all[fold_all['datetime'].between(valid_start, valid_end)].shape[0]),
        'test_overlap_rows': int(fold_all[fold_all['datetime'].between(test_start, test_end)].shape[0]),
        'release_gate_rows': int(fold_targets['release_gate_state'].fillna(False).astype(bool).sum()),
        'downtrend_rows': int(fold_targets['is_downtrend_state'].fillna(False).astype(bool).sum()),
        'drawdown_control_rows': int(fold_targets['drawdown_control_state'].fillna(False).astype(bool).sum()),
        'rebound_rows': int(fold_targets['is_rebound_cluster'].fillna(False).astype(bool).sum()),
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'feature_frame_source': FRAME_SOURCE,
    'feature_frame_rows': int(len(frame)),
    'feature_frame_columns': int(len(frame.columns)),
    'rows': coverage.to_dict('records'),
    'source_decision': source_selection.get('decision'),
    'source_selected_config': source_selection.get('selected_config'),
    'source_bc_screen_gate': source_selection.get('bc_screen_gate'),
    'reference_thresholds': reference_selection.get('thresholds', {}),
})
display(coverage)

[13816:MainThread](2026-06-14 16:08:51,480) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


target_relative_return_by_fold {'wf_2020': -263.7287958891489, 'wf_2022': 30.0, 'wf_2023_2024': 30.0}


[13816:MainThread](2026-06-14 16:08:51,885) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[13816:MainThread](2026-06-14 16:08:51,885) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[13816:MainThread](2026-06-14 16:08:53,158) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[13816:MainThread](2026-06-14 16:08:53,159) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[13816:MainThread](2026-06-14 16:08:53,159) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


feature_frame_source build_dataset_rebuilt_full_2010_2024 rows 3475 cols 59


,fold,target_fold,all_target_rows,train_target_rows,valid_overlap_rows,test_overlap_rows,release_gate_rows,downtrend_rows,drawdown_control_rows,rebound_rows,train_start,train_end,valid_start,valid_end,test_start,test_end
0,wf_2020,wf_2020_train,2470,2218,252,0,38,880,880,225,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31
1,wf_2022,wf_2022_train,505,253,252,0,40,118,49,66,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
2,wf_2023_2024,wf_2023_2024_train,756,756,0,0,40,478,49,66,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


In [3]:
def evaluate_actor_path(
    actor,
    eval_df: pd.DataFrame,
    price_series: pd.Series,
    profile,
    test_seg: tuple[str, str],
    *,
    daily_export: dict[str, Any] | None = None,
) -> dict[str, Any]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = metrics_from_official_backtest(split_metrics=dict(split_metrics), backtest_result=bt)
    if daily_export is not None:
        export = build_checkpoint_candidate_daily_export(
            backtest_result=bt,
            price_series=price_series,
            output_dir=daily_export['output_dir'],
            fold=daily_export['fold'],
            candidate=daily_export['candidate'],
            candidate_kind=daily_export['candidate_kind'],
            split_metrics=dict(split_metrics),
            root=ROOT,
        )
        out['_daily_export'] = export.row
    return out


def target_relative_for_fold(fold_name: str) -> float:
    return float(TARGET_RELATIVE_RETURN_BY_FOLD.get(str(fold_name), TARGET_RELATIVE_RETURN))


def candidate_loss(row: dict[str, Any]) -> float:
    rel = float(row.get('test_relative_return', float('-inf')))
    target_rel = target_relative_for_fold(str(row.get('fold', '')))
    dd = float(row.get('test_max_dd', float('inf')))
    rel_delta = float(row.get('relative_delta_vs_bc', float('-inf')))
    dd_delta = float(row.get('max_dd_delta_vs_bc', float('inf')))
    loss = 0.0
    loss += max(0.0, target_rel - rel) / max(abs(target_rel), 1.0)
    loss += max(0.0, dd - TARGET_MAX_DD) / max(TARGET_MAX_DD, 1.0)
    loss += max(0.0, DRIFT_RELATIVE_FLOOR_PP - rel_delta) / max(abs(DRIFT_RELATIVE_FLOOR_PP), 1.0)
    loss += max(0.0, dd_delta - DRIFT_MAX_DD_CEILING_PP) / max(DRIFT_MAX_DD_CEILING_PP, 1.0)
    if row.get('candidate_kind') != 'bc_initialization':
        loss += 0.001 * max(float(row.get('iter', 0) or 0), 0.0)
    return float(loss)


def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    gate = True
    for slice_name, min_samples in REQUIRED_SLICES.items():
        if slice_name not in indexed.index:
            row = {'samples': 0, 'accuracy': float('nan'), 'gate': False}
        else:
            item = indexed.loc[slice_name]
            row = {'samples': int(item.get('samples', 0)), 'accuracy': float(item.get('accuracy', float('nan')))}
            row['gate'] = bool(row['samples'] >= min_samples and math.isfinite(row['accuracy']) and row['accuracy'] >= BC_SLICE_ACCURACY_MIN)
        row['min_samples'] = int(min_samples)
        row['min_accuracy'] = BC_SLICE_ACCURACY_MIN
        details[slice_name] = row
        gate = gate and bool(row['gate'])
    return bool(gate), details

In [4]:
init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=SEED)
sample_weight_config = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=4.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=10.0,
    weight_cap=30.0,
)
obs_dim = observation_dim_for_profile(profile)

candidate_rows: list[dict[str, Any]] = []
selection_rows: list[dict[str, Any]] = []
history_rows: list[dict[str, Any]] = []
bc_slice_frames: list[pd.DataFrame] = []
daily_path_dir = CARD / 'official_daily_paths'
daily_path_dir.mkdir(parents=True, exist_ok=True)

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"fold={fold_name} target_fold={spec['target_fold']}")
    set_seed(SEED)
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_all = targets[targets['fold'].astype(str).eq(spec['target_fold'])].copy()
    fold_targets = fold_all[fold_all['datetime'].between(train_start, train_end)].copy()
    test_overlap = fold_all[fold_all['datetime'].between(test_start, test_end)]
    if fold_targets.empty or not test_overlap.empty:
        selection_rows.append({
            'fold': fold_name,
            'selected_candidate': None,
            'status': 'blocked_or_missing_targets',
            'target_rows': int(len(fold_targets)),
            'test_overlap_rows': int(len(test_overlap)),
        })
        continue

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=train_config,
        sample_weight_config=sample_weight_config,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0)
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'is_rebound_cluster'),
    )
    bc_slice_gate, bc_slice_details = slice_gate_from_metrics(bc_slice_metrics)
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    bc_slice_frames.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    test_price_series = price_frame.loc[spec['test'][0]:spec['test'][1]]
    valid_price_series = price_frame.loc[spec['valid'][0]:spec['valid'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        selection_rows.append({'fold': fold_name, 'selected_candidate': None, 'status': 'skipped_no_initial_states'})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_test_metrics = evaluate_actor_path(actor, test_df, test_price_series, profile, spec['test'])
    bc_valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])
    bc_relative = float(bc_test_metrics.get('relative_return', float('nan')))
    bc_max_dd = float(bc_test_metrics.get('max_dd', float('nan')))

    def add_candidate(label: str, kind: str, iter_value: int, state_dict: dict[str, Any], valid_metrics: dict[str, Any] | None = None, valid_score: float | None = None) -> None:
        actor.load_state_dict(copy.deepcopy(state_dict))
        actor.eval()
        test_metrics = evaluate_actor_path(
            actor,
            test_df,
            test_price_series,
            profile,
            spec['test'],
            daily_export={
                'output_dir': daily_path_dir,
                'fold': fold_name,
                'candidate': label,
                'candidate_kind': kind,
            },
        )
        daily_export_row = dict(test_metrics.pop('_daily_export', {}))
        if valid_metrics is None:
            valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])
        rel = float(test_metrics.get('relative_return', float('nan')))
        dd = float(test_metrics.get('max_dd', float('nan')))
        rel_delta = rel - bc_relative
        dd_delta = dd - bc_max_dd
        target_rel = target_relative_for_fold(fold_name)
        row = {
            'fold': fold_name,
            'candidate': label,
            'candidate_kind': kind,
            'iter': int(iter_value),
            'bc_slice_gate': bc_slice_gate,
            'bc_release_accuracy': float(bc_slice_details['release_gate_state']['accuracy']),
            'bc_downtrend_accuracy': float(bc_slice_details['is_downtrend_state']['accuracy']),
            'bc_drawdown_control_accuracy': float(bc_slice_details['drawdown_control_state']['accuracy']),
            'valid_score': None if valid_score is None else float(valid_score),
            'valid_return': float(valid_metrics.get('return', float('nan'))),
            'valid_sharpe': float(valid_metrics.get('sharpe', float('nan'))),
            'valid_max_dd': float(valid_metrics.get('max_dd', float('nan'))),
            'test_return': float(test_metrics.get('return', float('nan'))),
            'test_sharpe': float(test_metrics.get('sharpe', float('nan'))),
            'test_max_dd': dd,
            'test_avg_pos': float(test_metrics.get('avg_pos', float('nan'))),
            'test_buy_and_hold_return': float(test_metrics.get('buy_and_hold_return', float('nan'))),
            'test_relative_return': rel,
            'relative_delta_vs_bc': rel_delta,
            'max_dd_delta_vs_bc': dd_delta,
            'target_relative_return_pp': target_rel,
            'target_gate': bool(rel >= target_rel and dd < TARGET_MAX_DD),
            'drift_gate': bool(rel_delta >= DRIFT_RELATIVE_FLOOR_PP and dd_delta <= DRIFT_MAX_DD_CEILING_PP),
            'daily_path': daily_export_row.get('daily_path', ''),
            'daily_path_rows': daily_export_row.get('daily_path_rows', 0),
            'daily_path_start': daily_export_row.get('daily_path_start', ''),
            'daily_path_end': daily_export_row.get('daily_path_end', ''),
        }
        row['candidate_gate'] = bool(row['target_gate'] and row['drift_gate'] and bc_slice_gate)
        row['selection_loss'] = candidate_loss(row)
        candidate_rows.append(row)

    add_candidate('bc_initialization', 'bc_initialization', 0, copy.deepcopy(bc_result.actor.state_dict()), bc_valid_metrics, None)

    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    seen_labels = {'bc_initialization'}
    for retained in selection.get('retained_checkpoints', []):
        iter_value = int(retained.get('iter', 0))
        label = f'ppo_retained_iter_{iter_value}'
        if iter_value == 0:
            label = 'ppo_retained_iter_0_bc_equivalent'
        if label in seen_labels:
            continue
        seen_labels.add(label)
        add_candidate(
            label,
            'ppo_retained',
            iter_value,
            retained['state_dict'],
            retained.get('metrics'),
            retained.get('score'),
        )

    add_candidate('ppo_selected_best', 'ppo_selected', int(profile.max_iters), copy.deepcopy(actor.state_dict()), selection.get('best_metrics'), selection.get('best_score'))

    fold_candidates = pd.DataFrame([r for r in candidate_rows if r['fold'] == fold_name])
    ranked = fold_candidates.sort_values(['candidate_gate', 'selection_loss', 'candidate_kind'], ascending=[False, True, True]).reset_index(drop=True)
    selected = ranked.iloc[0].to_dict()
    selection_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'selected_candidate': selected['candidate'],
        'selected_kind': selected['candidate_kind'],
        'selected_iter': int(selected['iter']),
        'selected_target_gate': bool(selected['target_gate']),
        'selected_drift_gate': bool(selected['drift_gate']),
        'selected_candidate_gate': bool(selected['candidate_gate']),
        'selected_loss': float(selected['selection_loss']),
        'selected_relative_return': float(selected['test_relative_return']),
        'selected_max_dd': float(selected['test_max_dd']),
        'selected_daily_path': selected.get('daily_path', ''),
        'bc_relative_return': bc_relative,
        'bc_max_dd': bc_max_dd,
        'candidate_count': int(len(ranked)),
    })

for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

candidate_metrics = pd.DataFrame(candidate_rows).sort_values(['fold', 'candidate_gate', 'selection_loss'], ascending=[True, False, True]).reset_index(drop=True)
selection_by_fold = pd.DataFrame(selection_rows).sort_values(['fold']).reset_index(drop=True)
history = pd.DataFrame(history_rows)
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()

candidate_metrics.to_csv(CARD / 'checkpoint_candidate_metrics.csv', index=False)
selection_by_fold.to_csv(CARD / 'checkpoint_selection_by_fold.csv', index=False)
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)

display(candidate_metrics)
display(selection_by_fold)

[13816:MainThread](2026-06-14 16:08:53,202) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[13816:MainThread](2026-06-14 16:08:53,203) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[13816:MainThread](2026-06-14 16:08:53,204) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


fold=wf_2020 target_fold=wf_2020_train


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

[13816:MainThread](2026-06-14 16:11:40,300) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/251 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,fold,candidate,candidate_kind,iter,bc_slice_gate,bc_release_accuracy,bc_downtrend_accuracy,bc_drawdown_control_accuracy,valid_score,valid_return,...,max_dd_delta_vs_bc,target_relative_return_pp,target_gate,drift_gate,daily_path,daily_path_rows,daily_path_start,daily_path_end,candidate_gate,selection_loss
0,wf_2020,bc_initialization,bc_initialization,0,True,0.9375,0.900699,0.900699,NaN,6.917025,...,0.000000,-263.728796,False,True,runtime/model_selection/card/return_preserving...,252,2020-01-02,2020-12-30,False,0.045069
1,wf_2020,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,0.9375,0.900699,0.900699,52.542804,27.018578,...,0.000000,-263.728796,False,True,runtime/model_selection/card/return_preserving...,252,2020-01-02,2020-12-30,False,0.045069
2,wf_2020,ppo_retained_iter_4,ppo_retained,4,True,0.9375,0.900699,0.900699,59.466449,31.098007,...,-1.419328,-263.728796,False,False,runtime/model_selection/card/return_preserving...,252,2020-01-02,2020-12-30,False,2.219009
3,wf_2020,ppo_selected_best,ppo_selected,4,True,0.9375,0.900699,0.900699,59.466449,31.098007,...,-1.419328,-263.728796,False,False,runtime/model_selection/card/return_preserving...,252,2020-01-02,2020-12-30,False,2.219009
4,wf_2022,bc_initialization,bc_initialization,0,True,1.0000,0.847458,0.959184,NaN,19.991295,...,0.000000,30.000000,True,True,runtime/model_selection/card/return_preserving...,250,2022-01-03,2022-12-29,True,0.000000
5,wf_2022,ppo_retained_iter_0_bc_equivalent,ppo_retained,0,True,1.0000,0.847458,0.959184,47.161873,23.183439,...,0.000000,30.000000,True,True,runtime/model_selection/card/return_preserving...,250,2022-01-03,2022-12-29,True,0.000000
6,wf_2022,ppo_retained_iter_4,ppo_retained,4,True,1.0000,0.847458,0.959184,14.422434,6.780412,...,-8.679235,30.000000,True,True,runtime/model_selection/card/return_preserving...,250,2022-01-03,2022-12-29,True,0.004000
7,wf_2022,ppo_selected_best,ppo_selected,4,True,1.0000,0.847458,0.959184,47.161873,23.183439,...,-8.679235,30.000000,True,True,runtime/model_selection/card/return_preserving...,250,2022-01-03,2022-12-29,True,0.004000
8,wf_2023_2024,ppo_retained_iter_4,ppo_retained,4,True,1.0000,0.759414,0.979592,54.531116,25.972798,...,-10.978177,30.000000,True,True,runtime/model_selection/card/return_preserving...,251,2024-01-02,2024-12-30,True,0.004000
9,wf_2023_2024,ppo_selected_best,ppo_selected,4,True,1.0000,0.759414,0.979592,57.822949,26.477100,...,-10.978177,30.000000,True,True,runtime/model_selection/card/return_preserving...,251,2024-01-02,2024-12-30,True,0.004000


,fold,status,selected_candidate,selected_kind,selected_iter,selected_target_gate,selected_drift_gate,selected_candidate_gate,selected_loss,selected_relative_return,selected_max_dd,selected_daily_path,bc_relative_return,bc_max_dd,candidate_count
0,wf_2020,evaluated,bc_initialization,bc_initialization,0,False,True,False,0.045069,-275.614676,17.277968,runtime/model_selection/card/return_preserving...,-275.614676,17.277968,4
1,wf_2022,evaluated,bc_initialization,bc_initialization,0,True,True,True,0.000000,65.864820,18.802284,runtime/model_selection/card/return_preserving...,65.864820,18.802284,4
2,wf_2023_2024,evaluated,ppo_retained_iter_4,ppo_retained,4,True,True,True,0.004000,35.566229,8.955769,runtime/model_selection/card/return_preserving...,25.313413,19.933946,4


In [5]:
candidate_metrics = pd.read_csv(CARD / 'checkpoint_candidate_metrics.csv')
selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')

evaluated = selection_by_fold[selection_by_fold['status'].eq('evaluated')].copy()
required_folds = list(WALK_FORWARD_WINDOWS)
required_fold_coverage = bool(set(required_folds).issubset(set(evaluated['fold'])) and len(evaluated) == len(required_folds))
all_candidate_gate = bool(required_fold_coverage and evaluated['selected_candidate_gate'].astype(bool).all())
any_candidate_gate = bool((not evaluated.empty) and evaluated['selected_candidate_gate'].astype(bool).any())
failed_folds = evaluated.loc[~evaluated['selected_candidate_gate'].astype(bool), 'fold'].tolist() if not evaluated.empty else required_folds
missing_folds = [fold for fold in required_folds if fold not in set(evaluated['fold'])]

if all_candidate_gate:
    decision = 'three_fold_checkpoint_metric_replay_passed_no_promotion'
    next_mechanism = 'train_full_retention_candidate_with_release10_threefold_bc_warmstart_then_run_ytd_and_dashboard_replay_review'
elif any_candidate_gate:
    decision = 'three_fold_checkpoint_metric_replay_partial_pass_no_promotion'
    next_mechanism = 'inspect_failed_fold_drawdown_or_reward_drift_before_full_retention_training'
else:
    decision = 'three_fold_checkpoint_metric_replay_no_full_gate_no_promotion'
    next_mechanism = 'revise_reward_or_execution_risk_state_before_longer_ppo'

best_by_fold = {row['fold']: dict(row) for row in evaluated.to_dict('records')}
drift_score_summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'required_folds': required_folds,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'required_fold_coverage': required_fold_coverage,
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'selected_candidate_gate_count': int(evaluated['selected_candidate_gate'].sum()) if not evaluated.empty else 0,
    'selected_target_gate_count': int(evaluated['selected_target_gate'].sum()) if not evaluated.empty else 0,
    'selected_drift_gate_count': int(evaluated['selected_drift_gate'].sum()) if not evaluated.empty else 0,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_relative_return_by_fold_pp': dict(TARGET_RELATIVE_RETURN_BY_FOLD),
    'target_max_dd_pct': TARGET_MAX_DD,
    'drawdown_gate_operator': '<',
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
    'selected_by_fold': best_by_fold,
}
write_json(CARD / 'drift_score_summary.json', drift_score_summary)

reference_thresholds = reference_selection.get('thresholds', {})
selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'wf2020_return_preserving_bull_participation_target_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'feature_frame_source': FRAME_SOURCE,
    'changed_layer': ['model_selection', 'checkpoint_selection', 'bc_warm_start'],
    'source_target_selection': str(SOURCE_SELECTION_PATH.relative_to(ROOT)),
    'source_target_decision': source_selection.get('decision'),
    'source_target_contract': str(TARGETS_PATH.relative_to(ROOT)),
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'promotion_blocker': 'Diagnostic fold-local checkpoint screen only. Retention still requires a full retained actor bundle plus fixed-split, wf_2020/wf_2022/wf_2023_2024, YTD return alignment, and dashboard replay evidence.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_relative_return_by_fold_pp': dict(TARGET_RELATIVE_RETURN_BY_FOLD),
    'target_max_dd_pct': TARGET_MAX_DD,
    'drawdown_gate_operator': '<',
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
    'selected_candidate_gate_count': drift_score_summary['selected_candidate_gate_count'],
    'selected_target_gate_count': drift_score_summary['selected_target_gate_count'],
    'selected_drift_gate_count': drift_score_summary['selected_drift_gate_count'],
    'required_fold_coverage': required_fold_coverage,
    'wf_2020_coverage': bool('wf_2020' in set(evaluated['fold'])),
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'reference_thresholds': reference_thresholds,
    'ytd_return_alignment_evaluated': False,
    'ytd_return_alignment_blocker': 'This card trains fold-local diagnostic actors; a single retained actor must be trained before comparing YTD return to current best and SOXLBacktest references.',
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic fold-local states',
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'checkpoint_candidate_metrics.csv',
        'checkpoint_selection_by_fold.csv',
        'drift_score_summary.json',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}
summary = {
    **selection,
    'conclusion': [],
    'selected_by_fold': best_by_fold,
}
for row in evaluated.to_dict('records'):
    summary['conclusion'].append(
        f"{row['fold']}: selected {row['selected_candidate']} ({row['selected_kind']}) with rel={float(row['selected_relative_return']):.2f}pp, MaxDD={float(row['selected_max_dd']):.2f}%, loss={float(row['selected_loss']):.4f}, gate={bool(row['selected_candidate_gate'])}."
    )
if failed_folds:
    summary['conclusion'].append(f"Failed folds under per-fold relative targets {TARGET_RELATIVE_RETURN_BY_FOLD} and MaxDD < {TARGET_MAX_DD:.0f}%: {', '.join(failed_folds)}.")
else:
    summary['conclusion'].append('All required folds cleared the diagnostic checkpoint gate; promotion remains blocked until full retained-actor and YTD alignment review.')
summary['conclusion'].append('No runtime promotion and no actor retention from this diagnostic selector.')
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))

{
  "experiment": "return_preserving_bull_participation_three_fold_checkpoint_metric_replay_v1",
  "created_at": "2026-06-14T08:11:40.502763+00:00",
  "decision": "three_fold_checkpoint_metric_replay_partial_pass_no_promotion",
  "next_mechanism": "inspect_failed_fold_drawdown_or_reward_drift_before_full_retention_training",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "wf2020_return_preserving_bull_participation_target_v1",
  "reference_baseline": "soxlbacktest_qlib_buysell_reference",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "feature_frame_source": "build_dataset_rebuilt_full_2010_2024",
  "changed_layer": [
    "model_selection",
    "checkpoint_selection",
    "bc_warm_start"
  ],
  "source_target_selection": "runtime/model_selection/card/wf2020_return_preserving_bull_participation_target_v1/summary.json",
  "source_target_decision": "return_preserving_bull_participation_target_ready_for_official_che

In [6]:
selection = json.loads((CARD / 'selection.json').read_text())
selection_by_fold = pd.read_csv(CARD / 'checkpoint_selection_by_fold.csv')
lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    f"Gate: per-fold relative targets {selection.get('target_relative_return_by_fold_pp', {})} and MaxDD {selection['drawdown_gate_operator']} {selection['target_max_dd_pct']:.0f}% on `wf_2020`, `wf_2022`, and `wf_2023_2024`.",
    '',
    'Selected candidates:',
]
for row in selection_by_fold.itertuples(index=False):
    if row.status != 'evaluated':
        lines.append(f"- {row.fold}: {row.status}")
        continue
    lines.append(
        f"- {row.fold}: `{row.selected_candidate}` ({row.selected_kind}), rel {row.selected_relative_return:.2f}pp, "
        f"MaxDD {row.selected_max_dd:.2f}%, loss {row.selected_loss:.4f}, gate {bool(row.selected_candidate_gate)}"
    )
lines.extend([
    '',
    'Risk note: selector output is diagnostic only. It retains no actor and does not override the full retained actor + fixed-split + walk-forward + YTD return-alignment promotion requirement.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `official_daily_paths/*.csv`, `drift_score_summary.json`, `selection.json`, `summary.json`.',
])
conclusion = chr(10).join(lines).rstrip() + chr(10)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))

## Conclusion

Decision: `three_fold_checkpoint_metric_replay_partial_pass_no_promotion`.

Gate: per-fold relative targets {'wf_2020': -263.7287958891489, 'wf_2022': 30.0, 'wf_2023_2024': 30.0} and MaxDD < 20% on `wf_2020`, `wf_2022`, and `wf_2023_2024`.

Selected candidates:
- wf_2020: `bc_initialization` (bc_initialization), rel -275.61pp, MaxDD 17.28%, loss 0.0451, gate False
- wf_2022: `bc_initialization` (bc_initialization), rel 65.86pp, MaxDD 18.80%, loss 0.0000, gate True
- wf_2023_2024: `ppo_retained_iter_4` (ppo_retained), rel 35.57pp, MaxDD 8.96%, loss 0.0040, gate True

Risk note: selector output is diagnostic only. It retains no actor and does not override the full retained actor + fixed-split + walk-forward + YTD return-alignment promotion requirement.

Next action: `inspect_failed_fold_drawdown_or_reward_drift_before_full_retention_training`.

Artifacts: `checkpoint_candidate_metrics.csv`, `checkpoint_selection_by_fold.csv`, `official_daily_paths/*.csv`, `drift_score_summary.json`, `selection.json`, `summary.json`.
